# cross-product-normal — ex1: unit surface normal of a triangle via cross product

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cross-product-normal`. Running the final beacon cell reports progress against the `Geometry: Cross-product surface normal` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Cross-product surface normal` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-product-normal`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-product-normal"
DD_SUBTOPIC = "Geometry: Cross-product surface normal"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Surface normal via cross product — quick refresher

Given three triangle vertices `P1, P2, P3` (3-D), the surface normal is:
```
n = (P2 - P1) × (P3 - P1)
n_hat = n / ||n||
```
**Right-hand rule.** Curling the fingers from `(P2 - P1)` toward `(P3 - P1)` makes the thumb point along `n`. Swap the order of the two edges → normal flips sign.

**Degenerate case.** If the three vertices are colinear, the cross product is the zero vector and `||n|| = 0` — normalization gives nan/inf. Real renderers check `||n|| > eps` before normalizing.

**Why this matters.** Lighting (`L · n`), backface culling (`view · n < 0`), and ray-triangle intersection all need a consistent surface normal.

### Exercise 1 — unit surface normal of a triangle via cross product

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `cross(P2-P1, P3-P1)` followed by normalization to compute the unit surface normal of a triangle in 3-D, returning a `(3,)` tensor.
> Keywords: cross-product, normal, triangle, rendering
> ```

**KCs targeted:** `edges-from-shared-vertex`, `cross-then-normalize`

Implement `ex1_triangle_normal(P1, P2, P3)`:

1. Compute edges from the shared vertex `P1`:
   ```python
   e1 = P2 - P1
   e2 = P3 - P1
   ```
2. Cross product:
   ```python
   n = t.linalg.cross(e1, e2)
   ```
3. Normalize:
   ```python
   return n / n.norm()
   ```

Inputs: three `(3,)` float tensors. Output: `(3,)` float unit vector.

**Order matters.** `cross(e1, e2)` and `cross(e2, e1)` differ in sign. The drill convention is `cross(P2-P1, P3-P1)` — counter-clockwise winding when viewed from the front.

In [ ]:
def ex1_triangle_normal(P1: Tensor, P2: Tensor, P3: Tensor) -> Tensor:
    """Unit surface normal of triangle (P1, P2, P3) via cross product."""
    raise NotImplementedError()


def _test_ex1():
    # Case 1: triangle in the z=0 plane, CCW → normal points +z.
    P1 = t.tensor([0.0, 0.0, 0.0])
    P2 = t.tensor([1.0, 0.0, 0.0])
    P3 = t.tensor([0.0, 1.0, 0.0])
    n = ex1_triangle_normal(P1, P2, P3)
    assert n.shape == (3,), f'shape: {tuple(n.shape)}'
    expected = t.tensor([0.0, 0.0, 1.0])
    assert t.allclose(n, expected, atol=1e-5), f'expected +z normal, got {n}'
    assert abs(n.norm().item() - 1.0) < 1e-5, f'must be unit length, got {n.norm().item()}'

    # Case 2: swap P2/P3 → CW winding → normal points -z.
    n_flipped = ex1_triangle_normal(P1, P3, P2)
    assert t.allclose(n_flipped, t.tensor([0.0, 0.0, -1.0]), atol=1e-5), (
        f'CW winding should flip the normal, got {n_flipped}'
    )

    # Case 3: triangle in the x=5 plane → normal is +/- x.
    P1 = t.tensor([5.0, 0.0, 0.0])
    P2 = t.tensor([5.0, 1.0, 0.0])
    P3 = t.tensor([5.0, 0.0, 1.0])
    n = ex1_triangle_normal(P1, P2, P3)
    # (P2-P1)=(0,1,0); (P3-P1)=(0,0,1); cross = (1,0,0). Normal = +x.
    assert t.allclose(n, t.tensor([1.0, 0.0, 0.0]), atol=1e-5), f'expected +x, got {n}'

    # Case 4: tilted triangle — verify orthogonality to both edges.
    P1 = t.tensor([0.0, 0.0, 0.0])
    P2 = t.tensor([2.0, 1.0, 0.0])
    P3 = t.tensor([0.0, 1.0, 3.0])
    n = ex1_triangle_normal(P1, P2, P3)
    e1 = P2 - P1
    e2 = P3 - P1
    assert abs((n * e1).sum().item()) < 1e-5, f'n must be perp to e1, dot = {(n*e1).sum().item()}'
    assert abs((n * e2).sum().item()) < 1e-5, f'n must be perp to e2, dot = {(n*e2).sum().item()}'
    assert abs(n.norm().item() - 1.0) < 1e-5, 'unit length'

    # --- Visualization: triangle + outward normal in 3-D ---
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    P1 = t.tensor([0.0, 0.0, 0.0])
    P2 = t.tensor([2.0, 0.0, 0.0])
    P3 = t.tensor([0.5, 2.0, 0.0])
    n = ex1_triangle_normal(P1, P2, P3)
    centroid = (P1 + P2 + P3) / 3
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    # triangle as a closed loop
    tri = t.stack([P1, P2, P3, P1])
    ax.plot(tri[:, 0], tri[:, 1], tri[:, 2], 'b-', linewidth=2)
    ax.scatter(*[tri[:3, i] for i in range(3)], c='blue', s=50)
    # normal arrow from centroid
    ax.quiver(centroid[0], centroid[1], centroid[2],
              n[0], n[1], n[2], length=1.0, color='red', label='unit normal')
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5); ax.set_zlim(-0.5, 1.5)
    ax.set_title('ex1 — triangle + outward unit normal')
    ax.legend()
    plt.tight_layout(); plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_triangle_normal(P1: Tensor, P2: Tensor, P3: Tensor) -> Tensor:
    e1 = P2 - P1
    e2 = P3 - P1
    n = t.linalg.cross(e1, e2)
    return n / n.norm()
```

**Why `t.linalg.cross` and not `t.cross`.** `t.cross` requires a `dim` argument in modern PyTorch (it used to default to dim=-1 for size-3 tensors, but that default emits a deprecation warning since 1.8). `t.linalg.cross` is the explicit replacement.

**Why edges share `P1`.** You can also compute `cross(P2-P1, P3-P2)` — the cross product is the same up to sign (vectors are coplanar). But `P1` as the shared vertex is the convention because it matches how barycentric coordinates are defined.

**Robustness.** Real renderers compute `n.norm()` first; if `||n|| < 1e-8`, treat as a degenerate triangle and skip it. Without that guard, division by ~0 produces `inf` or `nan` normals which poison the lighting calculation downstream.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()